# Capítulo 5 — Selenium: Historia, Arquitectura y Comparativa con Playwright

> *«Selenium no es una herramienta anticuada; es la herramienta que definió lo que significa automatizar un navegador. Entenderla es entender los fundamentos de las pruebas E2E modernas.»*

**Objetivo:** Comprender la historia y evolución de Selenium desde 2004 hasta hoy, entender su arquitectura técnica (WebDriver W3C), aprender a escribir tests con Selenium 4 en Python, y realizar una comparativa honesta con Playwright para saber cuándo elegir cada herramienta.

---

## 1. La historia de Selenium: 20 años de automatización web

Selenium no surgió como un proyecto de testing académico. Nació de una necesidad real en una empresa de software.

### 2004 — El origen: ThoughtWorks y el JavaScript Runner

En 2004, **Jason Huggins** trabajaba en ThoughtWorks (una consultora de software) y necesitaba automatizar pruebas de regresión para una aplicación web interna. La solución que creó se llamó **JavaScriptTestRunner**: un script JavaScript que se inyectaba en el navegador y simulaba clics y escritura de texto.

El nombre «Selenium» surgió como broma interna: en química, el selenio es un antídoto para el envenenamiento por mercurio, y la herramienta de la competencia en ese momento era **Mercury Interactive** (fabricante de QuickTest Professional). El nombre quedó.

### 2006 — WebDriver: el enfoque nativo

**Simon Stewart** (Google) creó **WebDriver**, un enfoque completamente diferente: en lugar de inyectar JavaScript, WebDriver se comunicaba con el navegador a través de su **API nativa de automatización**. Cada navegador tenía su propio driver (ChromeDriver, GeckoDriver, etc.) que traducía los comandos de WebDriver en acciones reales del browser.

Ventaja clave: WebDriver no dependía de JavaScript, lo que lo hacía más robusto para casos complejos.

### 2011 — Selenium 2: la fusión

Selenium RC (el original) y WebDriver se fusionaron en **Selenium 2**, que unificó ambas aproximaciones bajo una sola API. En la práctica, WebDriver fue el ganador — el enfoque RC quedó como legacy.

### 2018 — Estandarización W3C

El **W3C** (World Wide Web Consortium) publicó el protocolo **WebDriver** como estándar oficial. Esto significó que todos los navegadores modernos debían implementar el protocolo de la misma manera. Ya no era solo un proyecto de Selenium — era un estándar de la web.

### 2021 — Selenium 4: modernización

**Selenium 4** incorporó:
- **CDP (Chrome DevTools Protocol):** acceso nativo a las DevTools del browser.
- **Relative Locators:** encontrar elementos en relación a otros (`above`, `below`, `near`).
- **Selenium Grid 4:** arquitectura renovada para ejecución paralela en la nube.
- **Nueva API de gestión de drivers:** ya no hace falta descargar ChromeDriver manualmente.

### Línea de tiempo visual

```
2004  Jason Huggins crea JavaScriptTestRunner en ThoughtWorks
  │
2006  Simon Stewart (Google) desarrolla WebDriver (API nativa)
  │
2008  Selenium RC lanzado como proyecto open source
  │
2011  Selenium 2: fusión de RC + WebDriver
  │
2013  ChromeDriver lanzado por Google
  │
2016  Selenium 3: soporte completo de WebDriver
  │
2018  W3C WebDriver Protocol se convierte en estándar oficial
  │
2020  Microsoft lanza Playwright (basado en CDP)
  │
2021  Selenium 4: CDP, Relative Locators, Grid modernizado
  │
2023  Selenium 4.x: soporte BiDi (bidireccional) emergente
```

---

## 2. Arquitectura técnica: el protocolo WebDriver W3C

Para entender cómo funciona Selenium 4, es fundamental entender el protocolo WebDriver.

### El flujo de una acción en Selenium

```
┌─────────────────────┐
│    Código Python     │  driver.find_element(...).click()
│    (tu test)         │
└──────────┬──────────┘
           │  Llamada al cliente Python de Selenium
┌──────────▼──────────┐
│  Selenium Python     │  Serializa la acción como JSON
│  Client Library      │  POST /session/{id}/element/{id}/click
└──────────┬──────────┘
           │  HTTP Request (REST API)
┌──────────▼──────────┐
│  WebDriver Server    │  ChromeDriver, GeckoDriver, etc.
│  (proceso separado)  │  Recibe el JSON, traduce a acción nativa
└──────────┬──────────┘
           │  Protocolo nativo del browser
┌──────────▼──────────┐
│  Navegador (Chrome,  │  Ejecuta el click real
│  Firefox, Edge...)   │
└─────────────────────┘
           │  Respuesta HTTP
           ▲  {"value": null} o error
```

**Implicación:** Cada acción individual (click, fill, find, get_text) es una **petición HTTP independiente**. Para una sola acción de usuario, pueden ser 3-5 peticiones HTTP:
1. `POST /session/{id}/element` → encontrar el elemento
2. `GET /session/{id}/element/{eid}/displayed` → verificar visibilidad
3. `POST /session/{id}/element/{eid}/click` → ejecutar el click

Esto es significativamente más lento que el protocolo WebSocket de Playwright.

### Cómo funciona Playwright (contraste)

```
┌─────────────────────┐
│    Código Python     │  page.locator(...).click()
└──────────┬──────────┘
           │  Llamada a Playwright Python
┌──────────▼──────────┐
│  Playwright Server   │  Proceso Node.js interno
│  (Python ↔ Node)     │
└──────────┬──────────┘
           │  WebSocket (conexión persistente)
┌──────────▼──────────┐
│  Browser (Chromium,  │  Conexión CDP directa
│  Firefox, WebKit)    │  Sin proceso intermediario separado
└─────────────────────┘
```

La conexión WebSocket es **persistente y bidireccional**. Playwright puede enviar múltiples comandos sin esperar respuestas intermedias, y el browser puede enviar eventos al test en tiempo real.

---

## 3. Instalación de Selenium 4 con WebDriver Manager

Selenium 4 ya no requiere descargar ChromeDriver manualmente. La biblioteca `webdriver-manager` gestiona esto automáticamente:

```bash
pip install selenium webdriver-manager
```

Con Selenium 4.6+, el `Service Manager` integrado puede manejar los drivers automáticamente:

```python
from selenium import webdriver

# Selenium 4.6+ — gestión automática de drivers
driver = webdriver.Chrome()  # descarga ChromeDriver automáticamente si es necesario

# Opción explícita con webdriver-manager
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
```

In [ ]:
# Verificar que Selenium está instalado
import selenium
print(f"Selenium versión: {selenium.__version__}")

# Verificar webdriver-manager
try:
    from webdriver_manager.chrome import ChromeDriverManager
    print("webdriver-manager disponible")
except ImportError:
    print("webdriver-manager no instalado — ejecuta: pip install webdriver-manager")

---

## 4. La API de Selenium 4: fundamentos

### 4.1 Tipos de locators en Selenium

Selenium usa la clase `By` para especificar cómo encontrar elementos:

| `By.*` | Descripción | Ejemplo |
|--------|-------------|------|
| `By.ID` | Por atributo `id` | `By.ID, "submit-btn"` |
| `By.NAME` | Por atributo `name` | `By.NAME, "username"` |
| `By.CSS_SELECTOR` | Por selector CSS | `By.CSS_SELECTOR, "[data-testid='btn-agregar']"` |
| `By.XPATH` | Por expresión XPath | `By.XPATH, "//button[@type='submit']"` |
| `By.CLASS_NAME` | Por clase CSS | `By.CLASS_NAME, "btn-complete"` |
| `By.TAG_NAME` | Por nombre de etiqueta | `By.TAG_NAME, "h1"` |
| `By.LINK_TEXT` | Por texto de enlace | `By.LINK_TEXT, "Iniciar sesión"` |

### 4.2 Esperas en Selenium (el punto más importante)

**Selenium no tiene auto-esperas.** Si el elemento no está en el DOM en el momento en que se ejecuta `find_element`, Selenium lanza `NoSuchElementException`. Hay tres tipos de esperas:

| Tipo | API | Descripción | Recomendación |
|------|-----|-------------|---------------|
| **Implícita** | `driver.implicitly_wait(5)` | Espera global para todos los find_element | Evitar — oculta problemas |
| **Explícita** | `WebDriverWait(driver, 10).until(EC...)` | Espera con condición específica | **Usar siempre** |
| **Fija** | `time.sleep(2)` | Pausa ciega | **Nunca usar en producción** |

Las esperas explícitas con `WebDriverWait` son el enfoque correcto:
```python
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, timeout=10)

# Espera a que el elemento sea clickable
btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='btn-agregar']")))
btn.click()

# Espera a que el texto sea visible
titulo = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-testid='tarea-titulo']")))
assert titulo.text == "Mi tarea"
```

In [ ]:
# ──────────────────────────────────────────────────────────────
# DEMO: Crear y completar una tarea con Selenium 4
# La app Flask debe estar corriendo en localhost:5000
# ──────────────────────────────────────────────────────────────
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

BASE_URL = "http://localhost:5000"

def limpiar_estado_selenium():
    import urllib.request
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )

# Configuración headless
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, timeout=10)

try:
    limpiar_estado_selenium()
    driver.get(BASE_URL)

    print("=== Test con Selenium 4 ===")

    # PASO 1: Crear tarea
    input_titulo = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='input-titulo']"))
    )
    input_titulo.clear()
    input_titulo.send_keys("Aprender Selenium 4")

    btn_agregar = driver.find_element(By.CSS_SELECTOR, "[data-testid='btn-agregar']")
    btn_agregar.click()

    # PASO 2: Esperar a que la tarea aparezca
    tarea_titulo = wait.until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-testid='tarea-titulo']"))
    )
    assert tarea_titulo.text.strip() == "Aprender Selenium 4", f"Esperado: 'Aprender Selenium 4', obtenido: '{tarea_titulo.text}'"
    print(f"✓ Paso 1: Tarea '{tarea_titulo.text}' creada")

    # PASO 3: Completar la tarea
    btn_completar = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='btn-completar']"))
    )
    btn_completar.click()

    # PASO 4: Verificar badge de completada
    badge = wait.until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-testid='badge-completada']"))
    )
    assert "Completada" in badge.text, f"Badge inesperado: '{badge.text}'"
    print(f"✓ Paso 2: Tarea completada — badge: '{badge.text}'")

    print("\nTest Selenium completado con éxito.")

finally:
    driver.quit()

---

## 5. Código equivalente: Selenium vs Playwright

Comparamos el mismo test en ambas herramientas para ver las diferencias concretas:

### Selenium 4
```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.add_argument("--headless=new")

driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, timeout=10)

driver.get("http://localhost:5000")

# Cada acción requiere espera explícita
inp = wait.until(EC.element_to_be_clickable(
    (By.CSS_SELECTOR, "[data-testid='input-titulo']")
))
inp.clear()
inp.send_keys("Mi tarea")

driver.find_element(By.CSS_SELECTOR, "[data-testid='btn-agregar']").click()

titulo = wait.until(EC.visibility_of_element_located(
    (By.CSS_SELECTOR, "[data-testid='tarea-titulo']")
))
assert titulo.text.strip() == "Mi tarea"

driver.quit()
```

### Playwright equivalente
```python
from playwright.sync_api import sync_playwright, expect

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("http://localhost:5000")

    # Auto-wait incluido — no necesita WebDriverWait
    page.fill("[data-testid='input-titulo']", "Mi tarea")
    page.click("[data-testid='btn-agregar']")

    # expect() con reintento automático
    expect(page.locator("[data-testid='tarea-titulo']").first).to_have_text("Mi tarea")

    browser.close()
```

**Observaciones:**
- Playwright es ~40% menos código para el mismo test.
- Selenium requiere `WebDriverWait` explícito en cada acción; Playwright lo incluye internamente.
- Playwright usa `expect()` con reintento automático; Selenium usa aserciones simples (que pueden fallar por timing).

---

## 6. Tabla comparativa completa

| Criterio | Selenium 4 | Playwright |
|----------|------------|------------|
| **Año de creación** | 2004 (Selenium RC), 2011 (WebDriver) | 2020 |
| **Protocolo** | HTTP REST (WebDriver W3C) | WebSocket (CDP) |
| **Auto-esperas** | ❌ Manuales (`WebDriverWait`) | ✅ Automáticas |
| **Verbosidad del código** | Alta | Baja |
| **Velocidad relativa** | Más lento (HTTP por acción) | Más rápido (WebSocket) |
| **Multi-browser** | Chrome, Firefox, Edge, Safari, IE | Chromium, Firefox, WebKit |
| **Soporte IE/Legacy** | ✅ Sí | ❌ No |
| **Trazas y vídeos** | Plugin externo necesario | ✅ Integrado |
| **Intercepción de red** | Limitada (via CDP addon) | ✅ Completa y sencilla |
| **Grid/Ejecución paralela** | ✅ Selenium Grid nativo | pytest-xdist / CI workers |
| **Soporte cloud** | BrowserStack, SauceLabs, LambdaTest | BrowserStack, SauceLabs |
| **Madurez y ecosistema** | 20+ años, muy estable | 4+ años, crecimiento rápido |
| **Curva de aprendizaje** | Media (esperas explícitas son complejas) | Baja (auto-wait simplifica mucho) |
| **Flakiness** | Mayor (sin auto-wait) | Menor (auto-wait reduce flakiness) |
| **Comunidad y docs** | Muy amplia | Creciendo rápidamente |

---

## 7. ¿Cuándo elegir Selenium?

A pesar de que Playwright es más moderno, hay escenarios donde Selenium sigue siendo la elección correcta:

### ✅ Usar Selenium cuando:

1. **Debes soportar Internet Explorer o browsers muy antiguos.** Playwright no soporta IE ni versiones antiguas de Edge. Selenium (vía IEDriver) sí.

2. **Tu proyecto ya tiene una inversión en Selenium.** Migrar una suite existente de 500 tests de Selenium a Playwright tiene un costo significativo que puede no justificarse.

3. **Necesitas Selenium Grid.** Para ejecución paralela masiva en infraestructura propia o cloud, Selenium Grid es maduro y battle-tested.

4. **Tu equipo tiene más experiencia con Selenium.** La curva de aprendizaje no es trivial para ninguna herramienta, y cambiar por cambiar puede reducir productividad.

5. **Necesitas soporte de lenguajes específicos.** Selenium tiene bindings oficiales para Java, Python, C#, Ruby, JavaScript y Kotlin. Playwright para Python, JavaScript, Java y C# — sin Ruby.

### ✅ Usar Playwright cuando:

1. **Proyecto nuevo.** Si estás empezando desde cero, Playwright ofrece mejor DX (Developer Experience) y menos flakiness.

2. **Necesitas interceptar o mockear peticiones de red.** Playwright tiene una API de intercepción de red excelente; en Selenium es mucho más complejo.

3. **La velocidad de la suite importa.** Playwright es significativamente más rápido para suites grandes.

4. **Quieres capacidades avanzadas de diagnóstico.** Las trazas, vídeos y screenshots integrados de Playwright son superiores.

5. **Pruebas de componentes o Single-Page Applications (SPA).** Playwright tiene mejor soporte para SPAs con React/Vue/Angular.

---

## 8. El futuro: WebDriver BiDi

El W3C está trabajando en una nueva especificación llamada **WebDriver BiDi** (Bidireccional), que combina lo mejor de ambos mundos:
- El estándar abierto de WebDriver.
- La comunicación bidireccional de CDP (vía WebSocket).

Con BiDi, Selenium podrá tener las capacidades de Playwright manteniendo su naturaleza de estándar abierto. Selenium 4.x ya empieza a implementar BiDi de forma experimental.

Esto sugiere que la brecha entre ambas herramientas se cerrará con el tiempo.

---

## 9. Ejercicios prácticos

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO 1: Implementar los mismos 3 tests en Selenium
# (crear, completar, eliminar)
# ──────────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import urllib.request

BASE_URL = "http://localhost:5000"

def limpiar():
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )

opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--no-sandbox")

driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, timeout=10)

try:
    # ── Test 1: Crear tarea ──────────────────────────────────
    limpiar()
    driver.get(BASE_URL)
    wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "[data-testid='input-titulo']")))\
        .send_keys("Test Selenium")
    driver.find_element(By.CSS_SELECTOR, "[data-testid='btn-agregar']").click()
    titulo = wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "[data-testid='tarea-titulo']")))
    assert titulo.text.strip() == "Test Selenium"
    print("✓ Test 1: Crear tarea — PASÓ")

    # ── Test 2: Completar tarea ──────────────────────────────
    btn_completar = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "[data-testid='btn-completar']")))
    btn_completar.click()
    badge = wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "[data-testid='badge-completada']")))
    assert "Completada" in badge.text
    print("✓ Test 2: Completar tarea — PASÓ")

    # ── Test 3: Eliminar tarea ───────────────────────────────
    btn_eliminar = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "[data-testid='btn-eliminar']")))
    btn_eliminar.click()
    msg_vacio = wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "[data-testid='msg-lista-vacia']")))
    assert msg_vacio.is_displayed()
    print("✓ Test 3: Eliminar tarea — PASÓ")

finally:
    driver.quit()

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO 2: Benchmark de velocidad — Selenium vs Playwright
# ──────────────────────────────────────────────────────────────
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from playwright.sync_api import sync_playwright
import urllib.request

def limpiar():
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )

def run_selenium(n_tareas=5):
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    driver = webdriver.Chrome(options=opts)
    wait = WebDriverWait(driver, timeout=10)
    try:
        limpiar()
        driver.get(BASE_URL)
        for i in range(n_tareas):
            inp = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "[data-testid='input-titulo']")))
            inp.clear()
            inp.send_keys(f"Tarea Selenium {i+1}")
            driver.find_element(
                By.CSS_SELECTOR, "[data-testid='btn-agregar']").click()
            wait.until(lambda d: len(d.find_elements(
                By.CSS_SELECTOR, "[data-testid='tarea-item']")) == i + 1)
    finally:
        driver.quit()

def run_playwright(n_tareas=5):
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True)
        page = browser.new_page()
        limpiar()
        page.goto(BASE_URL)
        for i in range(n_tareas):
            page.fill("[data-testid='input-titulo']", f"Tarea Playwright {i+1}")
            page.click("[data-testid='btn-agregar']")
            page.wait_for_function(
                f"document.querySelectorAll(\"[data-testid='tarea-item']\").length === {i+1}"
            )
        browser.close()

N = 5
print(f"Benchmark: crear {N} tareas con cada herramienta\n")

t0 = time.time()
run_selenium(N)
t_selenium = time.time() - t0
print(f"Selenium:   {t_selenium:.2f}s")

t0 = time.time()
run_playwright(N)
t_playwright = time.time() - t0
print(f"Playwright: {t_playwright:.2f}s")

if t_playwright > 0:
    ratio = t_selenium / t_playwright
    print(f"\nSelenium fue {ratio:.1f}x {'más lento' if ratio > 1 else 'más rápido'} que Playwright")

---

## 10. Preguntas para el informe

1. **Historia y protocolo:** En tus propias palabras, explica por qué el protocolo HTTP de Selenium es inherentemente más lento que el WebSocket de Playwright. ¿Qué implicaciones tiene esto para una suite de 500 tests que se ejecuta en CI/CD varias veces al día?

2. **Benchmark personal:** Ejecuta el ejercicio 2 (benchmark) y registra los tiempos. ¿Los resultados concuerdan con lo esperado teóricamente? Si hay diferencias, ¿a qué las atribuyes?

3. **Esperas explícitas vs auto-wait:** Identifica en el código de Selenium del Ejercicio 1 todas las líneas que son esperas explícitas (`WebDriverWait`, `until`, `EC.*`). ¿Cuántas son? ¿Cómo se simplificaría el código si Selenium tuviera auto-wait como Playwright?

4. **Decisión técnica:** Tu empresa tiene una suite de 200 tests escritos en Selenium 3. El gerente te pide evaluar si migrar a Playwright. Escribe una recomendación técnica de máximo 200 palabras considerando: costo de migración, beneficios esperados, riesgos y alternativas.

5. **WebDriver BiDi:** Investiga brevemente qué es WebDriver BiDi y cómo podría cambiar la comparativa Selenium vs Playwright en los próximos 2-3 años. ¿Cambiaría tu recomendación de la pregunta anterior?